# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv('PRICE_DATA')
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)
print(f"Found {len(parquet_files)} parquet files")


Found 3006 parquet files


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
# Write your code below.
def add_features(df):
    df = df.sort_values('Date')
    df['Close_lag_1'] = df.groupby('ticker')['Close'].shift(1)
    df['Adj_Close_lag_1'] = df.groupby('ticker')['Adj Close'].shift(1)
    df['returns'] = (df['Close'] / df['Close_lag_1']) - 1
    df['hi_lo_range'] = df['High'] - df['Low']
    return df

dd_feat = dd.read_parquet(parquet_files, index=False)
dd_feat = dd_feat.map_partitions(add_features)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [5]:
# Write your code below.
# Convert the Dask dataframe to pandas
df_feat = dd_feat.compute().reset_index(drop=True)

# Moving average of returns over a 10-day window, per ticker
df_feat["moving_avg_returns"] = (
    df_feat.groupby("ticker")["returns"]
    .transform(lambda x: x.rolling(10).mean())
)

In [6]:
# Check final dataframe
df_feat[df_feat['ticker'] == df_feat['ticker'].iloc[0]][
    ['Date', 'ticker', 'Close', 'Close_lag_1', 'returns', 'moving_avg_returns']
].head(15)

,Date,ticker,Close,Close_lag_1,returns,moving_avg_returns
0,1999-11-18,A,31.473534,NaN,NaN,NaN
1,1999-11-19,A,28.880543,31.473534,-0.082386,NaN
2,1999-11-22,A,31.473534,28.880543,0.089783,NaN
3,1999-11-23,A,28.612303,31.473534,-0.090909,NaN
4,1999-11-24,A,29.372318,28.612303,0.026563,NaN
5,1999-11-26,A,29.461731,29.372318,0.003044,NaN
6,1999-11-29,A,30.132332,29.461731,0.022762,NaN
7,1999-11-30,A,30.177038,30.132332,0.001484,NaN
8,1999-12-01,A,30.713520,30.177038,0.017778,NaN
9,1999-12-02,A,31.562946,30.713520,0.027656,NaN


Please comment:

+ *Was it necessary to convert to pandas to calculate the moving average return?*<br>
**Answer:** No. Dask supports .rolling().mean() so the moving average could have been calculated without converting to pandas.<br>

+ *Would it have been better to do it in Dask? Why?*<br>
**Answer:** Yes. Dask processes data in partitions without loading everything into memory at once. Pandas loads the entire dataset into memory, which could be problematic with very large datasets.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.